# Calc Flow DataFusion quickstart

Create an immutable Arrow-backed batch, compile a DataFusion pipeline, and inspect its output and metrics.

In [ ]:
import pyarrow as pa

from calc_flow import Batch, ExpressionOperator, Pipeline

In [ ]:
orders = Batch.table(
    pa.table(
        {
            "order_id": ["A-100", "A-101", "A-102"],
            "quantity": [3, 1, 4],
            "unit_price": [10, 12, 10],
        }
    )
)
orders.table_payload

In [ ]:
plan = (
    Pipeline("notebook-quickstart")
    .then(ExpressionOperator("calculate_gross", "gross = quantity * unit_price"))
    .then(
        ExpressionOperator(
            "large_orders",
            select=("order_id", "gross"),
            filter_expression="gross >= 20",
        )
    )
    .compile()
)
run = plan.execute({"input": orders})

In [ ]:
run.output.table_payload.to_pylist()

In [ ]:
{node_id: timing.duration_ns for node_id, timing in run.node_timings.items()}